# VidyaAI: Fine-tuning Gemma 4 with Unsloth

Fine-tune Gemma 4 E4B for multilingual Indian education using Unsloth + QLoRA.

**Requirements:** Google Colab with T4 GPU (free tier works)

## Setup

In [ ]:
%%capture
!pip uninstall unsloth unsloth_zoo -y
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir "git+https://github.com/unslothai/unsloth-zoo.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

## 1. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

# Configuration
MODEL_NAME = "google/gemma-4-e2b"  # 2B params — fits in Colab free tier
MAX_SEQ_LENGTH = 2048
DTYPE = None  # Auto-detect
LOAD_IN_4BIT = True  # QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")

## 2. Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth optimization
    random_state=42,
)

# Print trainable parameters
model.print_trainable_parameters()

## 3. Prepare Dataset

In [ ]:
from datasets import load_dataset, Dataset
import json

HINDI_SYSTEM = "तुम VidyaAI हो, एक बुद्धिमान शिक्षा सहायक। NCERT पाठ्यक्रम के अनुसार सरल हिंदी में जवाब दो। उदाहरणों का उपयोग करो।"
ENGLISH_SYSTEM = "You are VidyaAI, an intelligent education assistant. Answer according to NCERT curriculum in clear, simple English with examples."

def format_to_gemma(instruction, input_text, output_text):
    return (
        f"<start_of_turn>system\n{instruction}<end_of_turn>\n"
        f"<start_of_turn>user\n{input_text}<end_of_turn>\n"
        f"<start_of_turn>model\n{output_text}<end_of_turn>"
    )

# ── Source 1: MLQA Hindi (Facebook's multilingual QA) ──
print("Loading MLQA Hindi...")
mlqa = load_dataset("facebook/mlqa", "mlqa-translate-train.hi", split="train[:3000]")
mlqa_examples = []
for row in mlqa:
    q = row.get("question", "").strip()
    answers = row.get("answers", {})
    a_list = answers.get("text", []) if answers else []
    a = a_list[0].strip() if a_list else ""
    ctx = (row.get("context", "") or "")[:300].strip()
    if q and a:
        full_answer = f"{a}\n\nसंदर्भ: {ctx}" if ctx else a
        mlqa_examples.append({"text": format_to_gemma(HINDI_SYSTEM, q, full_answer)})
print(f"  MLQA Hindi: {len(mlqa_examples)} examples")

# ── Source 2: SQuAD English QA ──
print("Loading SQuAD...")
squad = load_dataset("rajpurkar/squad_v2", split="train[:5000]")
squad_examples = []
for row in squad:
    q = row.get("question", "").strip()
    answers = row.get("answers", {})
    a_list = answers.get("text", []) if answers else []
    a = a_list[0].strip() if a_list else ""
    ctx = (row.get("context", "") or "")[:300].strip()
    if q and a:
        full_answer = f"{a}\n\nContext: {ctx}" if ctx else a
        squad_examples.append({"text": format_to_gemma(ENGLISH_SYSTEM, q, full_answer)})
print(f"  SQuAD: {len(squad_examples)} examples")

# ── Source 3: Hand-crafted NCERT seed examples (high quality) ──
SEED = [
    (HINDI_SYSTEM, "प्रकाश संश्लेषण क्या है?",
     "प्रकाश संश्लेषण वह प्रक्रिया है जिसमें हरे पौधे सूर्य के प्रकाश की ऊर्जा का उपयोग करके कार्बन डाइऑक्साइड और पानी से ग्लूकोज बनाते हैं। इस प्रक्रिया में ऑक्सीजन भी निकलती है।\n\nसमीकरण: 6CO₂ + 6H₂O + सूर्य का प्रकाश → C₆H₁₂O₆ + 6O₂\n\nयह प्रक्रिया पत्तियों में मौजूद क्लोरोफिल की मदद से होती है।"),
    (HINDI_SYSTEM, "पाइथागोरस प्रमेय क्या है? उदाहरण के साथ समझाइए।",
     "पाइथागोरस प्रमेय कहता है कि एक समकोण त्रिभुज में, कर्ण का वर्ग अन्य दो भुजाओं के वर्गों के योग के बराबर होता है।\n\nसूत्र: c² = a² + b²\n\nउदाहरण: अगर दो भुजाएं 3 cm और 4 cm हैं:\nc² = 3² + 4² = 9 + 16 = 25\nc = √25 = 5 cm"),
    (HINDI_SYSTEM, "न्यूटन के गति के नियम बताइए।",
     "न्यूटन के तीन नियम:\n\n1. जड़त्व का नियम: वस्तु अपनी अवस्था में रहती है जब तक बाहरी बल न लगे। उदाहरण: बस रुकने पर आगे गिरना।\n\n2. F = ma: बल = द्रव्यमान × त्वरण। उदाहरण: भारी गेंद को फेंकने में ज़्यादा बल लगता है।\n\n3. क्रिया-प्रतिक्रिया: हर क्रिया की बराबर और विपरीत प्रतिक्रिया होती है। उदाहरण: रॉकेट का उड़ना।"),
    (HINDI_SYSTEM, "भारत की आज़ादी कब और कैसे मिली?",
     "भारत को 15 अगस्त 1947 को ब्रिटिश शासन से आज़ादी मिली।\n\nमुख्य बिंदु:\n1. महात्मा गांधी ने अहिंसा और सत्याग्रह के माध्यम से स्वतंत्रता आंदोलन का नेतृत्व किया।\n2. 1942 में 'भारत छोड़ो आंदोलन' ने ब्रिटिश शासन को हिला दिया।\n3. सुभाष चंद्र बोस ने आज़ाद हिंद फ़ौज बनाई।\n4. अंततः माउंटबेटन योजना के तहत भारत और पाकिस्तान का विभाजन हुआ।\n\nपंडित जवाहरलाल नेहरू भारत के पहले प्रधानमंत्री बने।"),
    (HINDI_SYSTEM, "कोशिका क्या है? इसके प्रकार बताइए।",
     "कोशिका जीवन की सबसे छोटी इकाई है। सभी जीवित प्राणी कोशिकाओं से बने हैं।\n\nदो मुख्य प्रकार:\n\n1. प्रोकैरियोटिक कोशिका: इसमें केंद्रक नहीं होता। उदाहरण: बैक्टीरिया।\n\n2. यूकैरियोटिक कोशिका: इसमें केंद्रक होता है। उदाहरण: पौधे और जानवरों की कोशिकाएं।\n\nमुख्य भाग: कोशिका झिल्ली, कोशिका द्रव्य, केंद्रक, माइटोकॉन्ड्रिया।"),
    (HINDI_SYSTEM, "अम्ल और क्षार में क्या अंतर है?",
     "अम्ल (Acid):\n- स्वाद खट्टा होता है\n- pH 7 से कम\n- नीले लिटमस को लाल करता है\n- उदाहरण: नींबू का रस (सिट्रिक अम्ल), सिरका (एसिटिक अम्ल)\n\nक्षार (Base):\n- स्वाद कड़वा, छूने में चिकना\n- pH 7 से अधिक\n- लाल लिटमस को नीला करता है\n- उदाहरण: साबुन, बेकिंग सोडा\n\nजब अम्ल और क्षार मिलते हैं तो उदासीनीकरण होता है: अम्ल + क्षार → लवण + जल"),
    (HINDI_SYSTEM, "भारत के मौलिक अधिकार क्या हैं?",
     "भारतीय संविधान में 6 मौलिक अधिकार हैं:\n\n1. समानता का अधिकार (अनुच्छेद 14-18): कानून के समक्ष समानता\n2. स्वतंत्रता का अधिकार (अनुच्छेद 19-22): भाषण, आवागमन की स्वतंत्रता\n3. शोषण के विरुद्ध अधिकार (अनुच्छेद 23-24): बाल श्रम निषेध\n4. धार्मिक स्वतंत्रता का अधिकार (अनुच्छेद 25-28)\n5. सांस्कृतिक और शैक्षिक अधिकार (अनुच्छेद 29-30)\n6. संवैधानिक उपचारों का अधिकार (अनुच्छेद 32): डॉ. अंबेडकर ने इसे संविधान की आत्मा कहा।"),
    (HINDI_SYSTEM, "सौरमंडल में कितने ग्रह हैं? उनके नाम बताइए।",
     "सौरमंडल में 8 ग्रह हैं (सूर्य से दूरी के क्रम में):\n\n1. बुध (Mercury) - सबसे छोटा, सूर्य के सबसे करीब\n2. शुक्र (Venus) - सबसे गर्म, पृथ्वी की जुड़वां\n3. पृथ्वी (Earth) - हमारा घर, जीवन वाला एकमात्र ग्रह\n4. मंगल (Mars) - लाल ग्रह\n5. बृहस्पति (Jupiter) - सबसे बड़ा ग्रह\n6. शनि (Saturn) - छल्लों वाला ग्रह\n7. अरुण (Uranus) - बगल में घूमता है\n8. वरुण (Neptune) - सबसे दूर, नीला ग्रह\n\nप्लूटो को 2006 में बौने ग्रह की श्रेणी में रखा गया।"),
    (ENGLISH_SYSTEM, "What is the water cycle?",
     "The water cycle (hydrological cycle) is the continuous movement of water on, above, and below Earth's surface.\n\nStages:\n1. Evaporation: Sun heats water in oceans/rivers, turning it into water vapor.\n2. Condensation: Water vapor rises, cools, and forms clouds.\n3. Precipitation: Water falls as rain, snow, or hail.\n4. Collection: Water flows into rivers, lakes, and oceans.\n\nThis cycle is essential for life and has been happening for billions of years."),
    (ENGLISH_SYSTEM, "Explain the concept of democracy.",
     "Democracy is a form of government where the people have the power to choose their leaders through free and fair elections.\n\nKey features:\n1. Universal adult suffrage: Every adult citizen has the right to vote.\n2. Rule of law: Everyone is equal before the law.\n3. Fundamental rights: Citizens have basic rights like freedom of speech.\n4. Independent judiciary: Courts work independently of the government.\n\nIndia is the world's largest democracy. Our Constitution, adopted on January 26, 1950, guarantees democratic governance."),
    (ENGLISH_SYSTEM, "What is photosynthesis? Explain with equation.",
     "Photosynthesis is the process by which green plants make food using sunlight.\n\nThe plant takes in carbon dioxide (CO₂) from air and water (H₂O) from soil. Using sunlight energy captured by chlorophyll in leaves, it converts them into glucose (food) and releases oxygen.\n\nEquation: 6CO₂ + 6H₂O + Sunlight → C₆H₁₂O₆ + 6O₂\n\nKey points:\n- Happens mainly in leaves (mesophyll cells)\n- Chlorophyll (green pigment) is essential\n- Oxygen is released as a byproduct\n- It is the basis of all food chains on Earth."),
    (ENGLISH_SYSTEM, "Explain Newton's three laws of motion with examples.",
     "Newton's Three Laws of Motion:\n\n1. First Law (Law of Inertia): An object stays at rest or in uniform motion unless acted on by an external force.\nExample: Passengers lurch forward when a bus stops suddenly.\n\n2. Second Law (F = ma): Force equals mass times acceleration. More force means more acceleration.\nExample: A cricket ball hit harder travels faster.\n\n3. Third Law (Action-Reaction): Every action has an equal and opposite reaction.\nExample: A rocket pushes gases downward, and the gases push the rocket upward.\n\nThese laws form the foundation of classical mechanics."),
    (ENGLISH_SYSTEM, "What is the difference between mitosis and meiosis?",
     "Mitosis:\n- Produces 2 identical daughter cells\n- Same chromosome number as parent (diploid)\n- For growth and repair\n- Happens in body (somatic) cells\n- One division cycle\n\nMeiosis:\n- Produces 4 genetically different cells\n- Half the chromosome number (haploid)\n- For producing sex cells (gametes)\n- Happens in reproductive organs\n- Two division cycles (meiosis I and II)\n- Crossing over creates genetic variation\n\nIn humans: Mitosis produces cells with 46 chromosomes, Meiosis produces gametes with 23 chromosomes."),
]

seed_examples = [{"text": format_to_gemma(s, i, o)} for s, i, o in SEED]
print(f"  Seed examples: {len(seed_examples)}")

# ── Combine all sources ──
all_examples = seed_examples + mlqa_examples + squad_examples
dataset = Dataset.from_list(all_examples).shuffle(seed=42)

print(f"\nTotal dataset: {len(dataset)} examples")
print(f"Sample:\n{dataset[0]['text'][:300]}...")

## 4. Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

In [ ]:
# Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_mem / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Memory reserved: {start_gpu_memory} GB / {max_memory} GB")

In [ ]:
# Train!
trainer_stats = trainer.train()
print(f"Training completed in {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## 5. Test the Fine-tuned Model

In [ ]:
FastLanguageModel.for_inference(model)

test_questions = [
    "प्रकाश संश्लेषण क्या है?",
    "भारत के पहले प्रधानमंत्री कौन थे?",
    "What is the formula for area of a circle?",
    "गुरुत्वाकर्षण बल क्या है?",
]

for q in test_questions:
    prompt = (
        "<start_of_turn>system\n"
        "तुम VidyaAI हो, एक बुद्धिमान शिक्षा सहायक। NCERT पाठ्यक्रम के अनुसार सरल हिंदी में जवाब दो।"
        "<end_of_turn>\n"
        f"<start_of_turn>user\n{q}<end_of_turn>\n"
        "<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
    )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"A: {response}")

## 6. Save & Upload to HuggingFace

In [ ]:
# Save LoRA adapter locally
model.save_pretrained("vidyaai-gemma4-lora")
tokenizer.save_pretrained("vidyaai-gemma4-lora")
print("Saved LoRA adapter to vidyaai-gemma4-lora/")

In [ ]:
# Upload to HuggingFace Hub
# First: huggingface-cli login
HF_REPO = "yashkuceriya/vidyaai-gemma4-lora"  # Change to your HF username

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"Uploaded to https://huggingface.co/{HF_REPO}")

In [ ]:
# Optional: Save as GGUF for Ollama deployment
model.save_pretrained_gguf(
    "vidyaai-gemma4-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("Saved GGUF model for Ollama deployment")